# Blackbox-explanation pipeline on POLAR simulated DTR data

Reproduces the SAE-based explanation pipeline from Explain_BB.pdf, adapted to the POLAR simulated 3-stage Dynamic Treatment Regime data.

**Setup**
- POLAR trajectory: `(s_1, a_1, s_2, a_2, s_3, a_3, s_4)`. `s_k \in [0,1]^2` are patient indicators, `a_k \in {0,1}` are treatment decisions, reward is at stage 3.
- **Blackbox M**: small MLP that maps the trajectory **through stage 3** (`s_1..s_3`, `a_1..a_3`) to a binary `outcome > median` label. Crucially, `s_4` is excluded from the input to keep the prediction non-trivial.
- **SAE**: top-k Sparse Autoencoder trained on M's second hidden layer.
- **LLM1**: proposes concept labels for each SAE feature using contrastive evidence sets.
- **Verifier**: short MATCH / NO_MATCH calls on held-out evidence.
- **LLM2**: structured explanation citing only active concepts.
- **Judge**: predicts the outcome from the explanation alone.
- **K-candidate selection**: K LLM2 outputs scored by `KL(P_M || P_Judge) + λ_spar |S| + λ_len Len(E)`.

**LLM backend**: `StubClient` by default (deterministic, no GPU). Switch to `QwenLocalClient` (cell 8) when you're ready to spend GPU time on real LLM calls.

## 1. Imports + config

In [1]:
import os, sys, json, warnings, time
import numpy as np
warnings.filterwarnings('ignore')

# Add the parent directory to sys.path so we can import bb_pipeline.*
sys.path.insert(0, os.path.abspath('..'))

from bb_pipeline.polar_data import generate_offline_data
from bb_pipeline.features import build_features, feature_names, compute_reward, reward_to_outcome_label
from bb_pipeline.blackbox import train_blackbox, collect_representations, auto_device
from bb_pipeline.sae import train_sae, encode_all
from bb_pipeline.evidence import collect_evidence, render_evidence_block
from bb_pipeline.llm import StubClient, QwenLocalClient, get_default_client
from bb_pipeline.llm1_concept import propose_concept
from bb_pipeline.verifier import score_concept, label_with_refinement
from bb_pipeline.llm2_explainer import explain_one
from bb_pipeline.judge import judge_explanation, kl_blackbox_judge
from bb_pipeline.pipeline import explain_with_selection

CONFIG = dict(
    n=10000,         # POLAR trajectories
    p=0.95,          # behavior-policy quality (0.5 random, 1.0 fully optimal)
    train_frac=0.8,
    mlp_hidden=64,
    mlp_dropout=0.1,
    mlp_weight_decay=1e-3,
    mlp_epochs=300,
    sae_latent=32,
    sae_k=8,
    sae_epochs=150,
    n_pos_train=5, n_neg_train=5,    # evidence shown to LLM1 (lean for speed)
    n_pos_val=8,   n_neg_val=8,      # held-out evidence for verifier (lean for speed)
    refinement_alpha=0.7,            # verifier-acc threshold below which we refine
    refinement_max_rounds=1,
    explain_K=3,                     # number of candidate explanations per input
    n_explain_inputs=2,              # how many test inputs to explain end-to-end
    seed=0,
)
print('device =', auto_device())
print(json.dumps({k:str(v) for k,v in CONFIG.items()}, indent=2))

device = mps
{
  "n": "10000",
  "p": "0.95",
  "train_frac": "0.8",
  "mlp_hidden": "64",
  "mlp_dropout": "0.1",
  "mlp_weight_decay": "0.001",
  "mlp_epochs": "300",
  "sae_latent": "32",
  "sae_k": "8",
  "sae_epochs": "150",
  "n_pos_train": "5",
  "n_neg_train": "5",
  "n_pos_val": "8",
  "n_neg_val": "8",
  "refinement_alpha": "0.7",
  "refinement_max_rounds": "1",
  "explain_K": "3",
  "n_explain_inputs": "2",
  "seed": "0"
}


## 2. Generate POLAR trajectories

Each row is `(s_1, a_1, s_2, a_2, s_3, a_3, s_4)`, 11 columns. Uses POLAR's behavior policy with quality `p`.

In [2]:
t = time.time()
D = generate_offline_data(num_samples=CONFIG['n'], p=CONFIG['p'], seed=CONFIG['seed'])
print(f'trajectories: {D.shape} in {time.time()-t:.1f}s')
print('first row:', np.round(D[0], 2))

trajectories: (10000, 11) in 0.1s
first row: [0.55 0.72 1.   0.47 0.81 1.   0.33 0.31 0.   0.55 0.69]


## 3. Derived features + outcome labels

- `X` (n, 28) = derived features through stage 3, no `s_4` (because `s_4` is the future we're predicting).
- `y` (n,) = `1[reward(s_3, s_4) > median]`.

In [3]:
X = build_features(D)
names = feature_names()
rewards = compute_reward(D)
y, threshold = reward_to_outcome_label(rewards)
print(f'X: {X.shape}  ({len(names)} named features)')
print(f'y: high/low balance = {int(y.sum())} / {len(y)}')
print(f'reward range: [{rewards.min():.2f}, {rewards.max():.2f}], threshold = {threshold:.2f}')
print('feature names:')
for i,nm in enumerate(names):
    print(f'  {i:>2d}  {nm}')

X: (10000, 28)  (28 named features)
y: high/low balance = 5000 / 10000
reward range: [-11.49, 14.25], threshold = 0.61
feature names:
   0  s1_ind1
   1  s1_ind2
   2  s2_ind1
   3  s2_ind2
   4  s3_ind1
   5  s3_ind2
   6  a1
   7  a2
   8  a3
   9  delta1_ind1
  10  delta1_ind2
  11  delta2_ind1
  12  delta2_ind2
  13  treatment_count
  14  all_treated
  15  never_treated
  16  switch_count
  17  early_treated
  18  mean_ind1
  19  mean_ind2
  20  max_ind1
  21  max_ind2
  22  min_ind1
  23  min_ind2
  24  s1_x_s3_ind1
  25  s1_x_s3_ind2
  26  ind1_range
  27  ind2_range


## 4. Train the MLP blackbox

Conservative sizing: `28 -> 64 -> 64 -> 2`, dropout 0.1, weight decay 1e-3, early stopping. The output `model.representation(x)` is the SAE's input.

In [4]:
rng = np.random.default_rng(CONFIG['seed'])
idx = rng.permutation(len(X))
n_train = int(CONFIG['train_frac'] * len(X))
tr, va = idx[:n_train], idx[n_train:]

t = time.time()
model, train_res, scaler = train_blackbox(
    X[tr], y[tr], X[va], y[va],
    hidden_dim=CONFIG['mlp_hidden'], epochs=CONFIG['mlp_epochs'],
    dropout=CONFIG['mlp_dropout'], weight_decay=CONFIG['mlp_weight_decay'],
    seed=CONFIG['seed'],
)
print(f'MLP trained in {time.time()-t:.1f}s')
print(f'  train_acc={train_res.train_acc:.3f}  val_acc={train_res.val_acc:.3f}  best_epoch={train_res.best_epoch}')
print(f'  overfit_warning={train_res.overfit_warning}')
if train_res.overfit_warning:
    print('  >> consider shrinking hidden_dim to 32 and re-running.')

MLP trained in 2.2s
  train_acc=0.907  val_acc=0.906  best_epoch=14
  overfit_warning=False


## 5. Train the SAE on the MLP hidden layer

Top-k SAE with auxiliary dead-feature loss (Gao et al. 2024). The number of *alive* features is bounded by the intrinsic complexity of what M actually encodes — we expect ~15 alive out of 32 for this DTR setup.

In [5]:
H_tr = collect_representations(model, X[tr], scaler)
H_va = collect_representations(model, X[va], scaler)
print(f'hidden reps: train={H_tr.shape}, val={H_va.shape}, mean nonzero frac={(H_tr>0).mean():.3f}')

t = time.time()
sae, sae_res = train_sae(
    H_tr, H_va,
    latent_dim=CONFIG['sae_latent'], k=CONFIG['sae_k'],
    epochs=CONFIG['sae_epochs'], seed=CONFIG['seed'],
)
alive = int((sae_res.feature_density >= 0.005).sum())
print(f'SAE trained in {time.time()-t:.1f}s')
print(f'  explained_variance={sae_res.explained_variance:.3f}')
print(f'  alive features (density >= 0.5%) = {alive} / {CONFIG["sae_latent"]}')
print(f'  per-feature density distribution:')
for i, d in enumerate(sae_res.feature_density):
    bar = '█' * int(d * 30)
    tag = ' (DEAD)' if d < 0.005 else ''
    print(f'    i{i:>2d}  {d:.3f}  {bar}{tag}')

hidden reps: train=(8000, 64), val=(2000, 64), mean nonzero frac=0.685


SAE trained in 69.2s
  explained_variance=0.987
  alive features (density >= 0.5%) = 18 / 32
  per-feature density distribution:
    i 0  0.475  ██████████████
    i 1  0.437  █████████████
    i 2  0.479  ██████████████
    i 3  0.419  ████████████
    i 4  0.531  ███████████████
    i 5  0.624  ██████████████████
    i 6  0.394  ███████████
    i 7  0.001   (DEAD)
    i 8  0.463  █████████████
    i 9  0.002   (DEAD)
    i10  0.000   (DEAD)
    i11  0.000   (DEAD)
    i12  0.000   (DEAD)
    i13  0.502  ███████████████
    i14  0.000   (DEAD)
    i15  0.496  ██████████████
    i16  0.000   (DEAD)
    i17  0.000   (DEAD)
    i18  0.000   (DEAD)
    i19  1.000  ██████████████████████████████
    i20  0.000   (DEAD)
    i21  0.000   (DEAD)
    i22  0.639  ███████████████████
    i23  0.000   (DEAD)
    i24  0.257  ███████
    i25  0.006  
    i26  0.338  ██████████
    i27  0.019  
    i28  0.001   (DEAD)
    i29  0.486  ██████████████
    i30  0.430  ████████████
    i31  0.000   (DEAD

## 6. Collect evidence sets per SAE feature

For each alive feature `i` we form `(E_i^+, E_i^-)`: top activators vs. zero activators. Train half is shown to LLM1 when proposing the concept; held-out half is used by the verifier.

In [6]:
Z_tr = encode_all(sae, H_tr)
Z_va = encode_all(sae, H_va)
X_tr_arr = X[tr]
X_va_arr = X[va]

evidence = collect_evidence(
    Z_tr,
    n_pos=CONFIG['n_pos_train'], n_neg=CONFIG['n_neg_train'],
    n_pos_val=CONFIG['n_pos_val'], n_neg_val=CONFIG['n_neg_val'],
    seed=CONFIG['seed'],
)
print(f'features with usable evidence: {len(evidence)} / {CONFIG["sae_latent"]}')
for fid in sorted(evidence)[:3]:
    e = evidence[fid]
    print(f'\n--- feature {fid} (density={e.density:.3f}) ---')
    block = render_evidence_block(
        X_tr_arr, names,
        e.pos_train_idx[:2], e.neg_train_idx[:2],
    )
    print(block)

features with usable evidence: 17 / 32

--- feature 0 (density=0.482) ---
### POSITIVE (feature strongly active)
Example 1:
  - treatment_count=+1.00
  - a3=+1.00
  - switch_count=+1.00
  - max_ind2=+0.87
  - s3_ind2=+0.87
  - s3_ind1=+0.86
  - max_ind1=+0.86
  - s1_ind2=+0.77
Example 2:
  - treatment_count=+2.00
  - switch_count=+2.00
  - a1=+1.00
  - early_treated=+1.00
  - a3=+1.00
  - s3_ind2=+0.90
  - max_ind2=+0.90
  - s3_ind1=+0.78

### NEGATIVE (feature near zero)
Example 1:
  - treatment_count=+1.00
  - early_treated=+1.00
  - switch_count=+1.00
  - a1=+1.00
  - max_ind1=+0.76
  - s1_ind1=+0.76
  - s2_ind1=+0.75
  - mean_ind1=+0.65
Example 2:
  - switch_count=+2.00
  - treatment_count=+1.00
  - a2=+1.00
  - max_ind1=+0.66
  - s1_ind1=+0.66
  - s2_ind1=+0.57
  - mean_ind1=+0.51
  - max_ind2=+0.42

--- feature 1 (density=0.426) ---
### POSITIVE (feature strongly active)
Example 1:
  - treatment_count=+2.00
  - switch_count=+2.00
  - a1=+1.00
  - a3=+1.00
  - early_treated=+1.00


## 7. LLM client

Default is `StubClient` — instant, deterministic, returns canned outputs that satisfy each prompt schema. Use it to verify pipeline plumbing in seconds.

To run with a real LLM, comment out the StubClient line and uncomment the QwenLocalClient line. Sizes:
- `Qwen/Qwen2.5-1.5B-Instruct` (~3 GB, safe on most GPUs)
- `Qwen/Qwen2.5-3B-Instruct` (~6 GB, better quality)
- `Qwen/Qwen2.5-7B-Instruct` (~14 GB)

In [7]:
# LLM1/LLM2/Judge run on Qwen 2.5-3B-Instruct (~6GB).
# First load downloads the model — this can take 5-10 min the first time.
from bb_pipeline.verifier import EmbeddingVerifier, LLMVerifier

client = QwenLocalClient(model_id='Qwen/Qwen2.5-3B-Instruct')
# client = QwenLocalClient(model_id='Qwen/Qwen2.5-1.5B-Instruct')
# client = StubClient()
verifier = EmbeddingVerifier(model_name='sentence-transformers/all-MiniLM-L6-v2')

print(f'LLM1/LLM2/Judge client: {client.name}')
print(f'verifier backend: {type(verifier).__name__}')
_ = client.chat([{'role':'user','content':'hi'}], max_new_tokens=4)
verifier._ensure_loaded()
print('all models warmed up')

LLM1/LLM2/Judge client: qwen-local
verifier backend: EmbeddingVerifier


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


all models warmed up


## 8. LLM1: propose concept labels with a TWO-PASS gold-buffer loop

PDF Step 6 (final paragraph) describes "gold buffer" — store high-quality (concept, evidence) records and prepend them as few-shot demonstrations to subsequent LLM1 calls.

Our two-pass implementation:

1. **Pass 1**: name every alive feature with no buffer (cold start). Concepts that hit `verifier_acc >= alpha` are pushed to the gold buffer.
2. **Pass 2**: for the remaining low-acc features, re-run LLM1 with `gold_k` worked examples from pass 1 as few-shot context, hoping the style guidance produces a better concept.

Final concept dict combines pass-1 winners + pass-2 (re)labels for the rest.

In [8]:
from bb_pipeline.gold_buffer import GoldBuffer

concept_labels = {}     # {feature_idx: concept string}
concept_records = {}    # {feature_idx: ConceptResult}
verifier_records = {}   # {feature_idx: VerifierResult}

# === PASS 1: cold start, no gold buffer ===
print('=== PASS 1: cold-start naming (no gold buffer) ===')
t0 = time.time()
gold = GoldBuffer(max_size=10)
pass1_failures = []
for fid in sorted(evidence):
    res, vres = label_with_refinement(
        client, evidence[fid], X_tr_arr, names,
        verifier=verifier,
        alpha=CONFIG['refinement_alpha'],
        max_rounds=CONFIG['refinement_max_rounds'],
    )
    concept_labels[fid] = res.concept
    concept_records[fid] = res
    verifier_records[fid] = vres
    tag = ''
    if vres.accuracy >= CONFIG['refinement_alpha']:
        gold.push(evidence[fid], res.concept, vres.accuracy)
        tag = '  -> pushed to gold buffer'
    else:
        pass1_failures.append(fid)
    print(f'  i{fid}: "{res.concept}"  acc={vres.accuracy:.2f}  rounds={len(res.refinement_history)-1}{tag}')

print(f'\nPass 1 done in {time.time()-t0:.1f}s')
print(f'Gold buffer size: {len(gold)}')
print(f'Pass 2 will re-label {len(pass1_failures)} features (acc < {CONFIG["refinement_alpha"]}).')

# === PASS 2: re-label failures with few-shot context from gold buffer ===
print('\n=== PASS 2: re-naming low-acc features with gold-buffer few-shot ===')
t1 = time.time()
pass2_improved = 0
for fid in pass1_failures:
    if len(gold) == 0:
        print(f'  i{fid}: skipped (gold buffer empty, no examples to learn from)')
        continue
    new_res, new_vres = label_with_refinement(
        client, evidence[fid], X_tr_arr, names,
        verifier=verifier,
        alpha=CONFIG['refinement_alpha'],
        max_rounds=CONFIG['refinement_max_rounds'],
        gold_buffer=gold, gold_k=2, gold_seed=fid,   # deterministic per feature
    )
    old_acc = verifier_records[fid].accuracy
    delta = new_vres.accuracy - old_acc
    keep = new_vres.accuracy >= old_acc
    keep_tag = 'kept' if keep else 'reverted'
    print(f'  i{fid}: "{new_res.concept}"  acc={new_vres.accuracy:.2f}  (was {old_acc:.2f}, Δ={delta:+.2f}, {keep_tag})')
    if keep:
        concept_labels[fid] = new_res.concept
        concept_records[fid] = new_res
        verifier_records[fid] = new_vres
        if new_vres.accuracy >= CONFIG['refinement_alpha']:
            pass2_improved += 1

print(f'\nPass 2 done in {time.time()-t1:.1f}s')
print(f'Features pushed above alpha by gold buffer: {pass2_improved} / {len(pass1_failures)}')
print(f'\nTOTAL LLM1 time: {time.time()-t0:.1f}s')
print(f'Final concept count above alpha: {sum(1 for v in verifier_records.values() if v.accuracy >= CONFIG["refinement_alpha"])} / {len(verifier_records)}')

=== PASS 1: cold-start naming (no gold buffer) ===


  i0: "Treatment diversity and switches"  acc=0.50  rounds=1


  i1: "MIXED"  acc=0.50  rounds=1


  i2: "MIXED"  acc=0.50  rounds=1


  i3: "Ind2 Range > 0.5"  acc=0.88  rounds=0  -> pushed to gold buffer


...

  i24: "Treatment count increase"  acc=0.88  (was 0.56, Δ=+0.31, kept)


  i25: ""Early treated increase""  acc=0.50  (was 0.38, Δ=+0.12, kept)


  i26: ""Max Ind1 < 0.9""  acc=0.62  (was 0.50, Δ=+0.12, kept)


  i27: ""Low switch_count range""  acc=0.50  (was 0.50, Δ=+0.00, kept)


  i29: "MIXED"  acc=0.81  (was 0.44, Δ=+0.38, kept)

Pass 2 done in 673.0s
Features pushed above alpha by gold buffer: 4 / 13

TOTAL LLM1 time: 1056.0s
Final concept count above alpha: 8 / 17


## 9. Pick test inputs and run end-to-end (LLM2 + Judge + K-selection)

For each chosen input from the validation set:
1. Get blackbox prediction `P_M(y | x)`.
2. Get active SAE features and their values.
3. Generate K candidate explanations (LLM2).
4. Score each with the Judge (KL + sparsity + length penalties).
5. Print the winning explanation.

In [ ]:
## 8b. Compute SAE feature directionality (polarity)
# For each labeled SAE feature, what does its activation say about the
# blackbox label distribution? We use point-biserial correlation between
# (z_i > 0) and y on the train set — see feature_polarity.py.
#
# This is then handed to LLM2 so it can write directionally faithful
# explanations (e.g., not claim a "[→ LOW]" feature supports HIGH outcome).
from bb_pipeline.feature_polarity import compute_polarities, render_polarity_tag

polarities = compute_polarities(Z_tr, y[tr], feature_indices=sorted(concept_labels.keys()))
print(f'computed polarity for {len(polarities)} features')
print(f'  HIGH-pushing: {sum(1 for p in polarities.values() if p.direction == "HIGH")}')
print(f'  LOW-pushing:  {sum(1 for p in polarities.values() if p.direction == "LOW")}')
print(f'  NEUTRAL:      {sum(1 for p in polarities.values() if p.direction == "NEUTRAL")}')

print('\nSample annotated dictionary entries:')
for fid in sorted(polarities)[:5]:
    print(f'  i{fid} -> "{concept_labels[fid]}" {render_polarity_tag(polarities[fid])}')

In [ ]:
## 8c. Snapshot the expensive state to disk
# After this cell you can quick-iterate the LLM2/Judge/loss side by reloading
# from disk instead of re-running the ~17min LLM1 phase.
#
# To resume:
#   from bb_pipeline.state_io import load_pipeline_state
#   from bb_pipeline.blackbox import OutcomeMLP
#   from bb_pipeline.sae import TopKSAE
#   state = load_pipeline_state('snapshot/run_001',
#                               mlp_class=OutcomeMLP, sae_class=TopKSAE)
#   model, sae, polarities = state['model'], state['sae'], state['polarities']
#   ... continue from cell 18.
from bb_pipeline.state_io import save_pipeline_state

snapshot_dir = 'snapshot/run_001'
save_pipeline_state(
    snapshot_dir,
    config=CONFIG,
    traj=D, X=X, y=y, tr_idx=tr, va_idx=va, scaler=scaler,
    model=model, sae=sae,
    H_tr=H_tr, H_va=H_va, Z_tr=Z_tr, Z_va=Z_va,
    evidence=evidence,
    concept_labels=concept_labels,
    verifier_records=verifier_records,
    polarities=polarities,
)

In [ ]:
import torch
device = next(model.parameters()).device
test_pick = rng.choice(len(va), size=CONFIG['n_explain_inputs'], replace=False)

results = []
for k_, vidx in enumerate(test_pick):
    x = X_va_arr[vidx]
    z = Z_va[vidx]
    active = [int(j) for j in np.where(z > 0)[0] if j in concept_labels]
    if not active:
        print(f'[skip {k_}] no labeled active features for this input')
        continue
    activ_vals = [float(z[j]) for j in active]
    x_norm = ((x - scaler['mean']) / scaler['std']).astype(np.float32)
    with torch.no_grad():
        p_m_raw = model.predict_proba(torch.from_numpy(x_norm).to(device).unsqueeze(0)).cpu().numpy().reshape(-1)
    p_m = np.array([float(p_m_raw[0]), float(p_m_raw[1])])
    print(f'\n=========================================================')
    print(f'input #{k_} (val idx {vidx})')
    print(f'  blackbox P(low, high) = ({p_m[0]:.3f}, {p_m[1]:.3f})')
    print(f'  active features = {active}')
    # Show polarities for the active features so we can sanity-check
    for j in active:
        if j in polarities:
            p = polarities[j]
            print(f'    i{j}: "{concept_labels[j]}" {render_polarity_tag(p)}')
    res = explain_with_selection(
        client, input_idx=int(vidx),
        p_blackbox=p_m,
        active_features=active, activation_values=activ_vals,
        concept_labels=concept_labels,
        polarities=polarities,
        K=CONFIG['explain_K'],
    )
    for c_i, c in enumerate(res.candidates):
        marker = '<-- WIN' if c_i == res.winner_index else ''
        print(f'  cand {c_i}: KL={c.kl:.3f}  |S|={c.sparsity}  Len={c.length}  total={c.total_loss:.3f}  Judge=({c.judge.prob_low:.2f},{c.judge.prob_high:.2f}) {marker}')
    print('  WINNING EXPLANATION:')
    print('  ' + res.winner.explanation.raw_text.replace('\n', '\n  '))
    results.append(res)

## 10. Summary

In [10]:
if results:
    kls = [r.winner.kl for r in results]
    print(f'#explained inputs: {len(results)}')
    print(f'mean winning KL(P_M || P_Judge): {np.mean(kls):.3f}')
    print(f'mean cited features per winner: {np.mean([r.winner.sparsity for r in results]):.1f}')
    print(f'mean mechanism count per winner: {np.mean([r.winner.length for r in results]):.1f}')

    # Verifier-acc summary
    accs = [v.accuracy for v in verifier_records.values()]
    print(f'\nverifier accuracy across {len(accs)} concepts: mean={np.mean(accs):.2f}, min={np.min(accs):.2f}, max={np.max(accs):.2f}')
    print(f'concepts above alpha={CONFIG["refinement_alpha"]}: {sum(a >= CONFIG["refinement_alpha"] for a in accs)} / {len(accs)}')
else:
    print('no inputs explained -- check that some active features have concept labels')

#explained inputs: 2
mean winning KL(P_M || P_Judge): 1.242
mean cited features per winner: 2.0
mean mechanism count per winner: 2.0

verifier accuracy across 17 concepts: mean=0.66, min=0.50, max=0.88
concepts above alpha=0.7: 8 / 17


## Next steps you might want

- **Switch to a real LLM** (cell 7) to see meaningful concept names and explanations.
- **Increase `n_explain_inputs`** for a real KL distribution.
- **Add a gold buffer** that stores winning records and prepends them as few-shot examples in subsequent LLM1 calls (PDF Step 6 final paragraph).
- **Try harder tasks**: drop more features (e.g., also drop `s_3` to predict from earlier history only) to push the MLP harder and produce more diverse SAE concepts.